In [1]:
import torch
import os
import ast
import re

## Log file helper functions

In [2]:
def get_config_dict(model, model_path, dataset="synthetic_res_scbm"):
    log_file = os.path.join("experiments", model, dataset, model_path, "log.txt")
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            lines = f.readlines()
            info_dict = lines[0].strip()  
            info_dict =  ast.literal_eval(info_dict)
            
        return info_dict
    else:
        raise FileNotFoundError(f"Log file not found: {log_file}")
    
def test_metrics(model, model_path, dataset="synthetic_res_scbm"):
    metrics = {}
    with open(os.path.join("experiments", model, dataset, model_path, "log.txt"), "r") as f:
        lines = f.readlines()
        for line in lines:
            if "Test" in line:
                test_line = line
                break
        y_accuracy = re.findall(r"y_accuracy:\s*([0-9.]+)", test_line)[0]
        c_accuracy = re.findall(r"c_accuracy:\s*([0-9.]+)", test_line)[0]
        c_auc = re.findall(r"c_AUROC:\s*([0-9.]+)", test_line)[0]
        metrics["y_accuracy"] = float(y_accuracy)
        metrics["c_accuracy"] = float(c_accuracy)
        metrics["c_auc"] = float(c_auc)
    return metrics   

def dataset_info(model, model_path, dataset="synthetic_res_scbm"):
    log_file_path = os.path.join("experiments", model, dataset, model_path, "log.txt")
    with open(log_file_path, "r") as f:
        lines = f.readlines()
        info = {}
        for line_num, line in enumerate(lines):
            if line_num == 0:
                continue
            if "Epoch" in line:
                break
            
            if ":" in line:
                key, value = line.split(":", 1)
                info[key.strip()] = value.strip()
    return info


def get_data_dir_path_from_model_log(model, model_path, dataset="synthetic_res_scbm"):
    log_file_path = os.path.join("experiments", model, dataset, model_path, "log.txt")
    with open(log_file_path, "r") as f:
        lines = f.readlines()
        for line in lines:
            if line.startswith("data_dir:"):
                data_dir_line = line
                break
        data_dir = data_dir_line.split("data_dir:")[1].strip()
        data_dir = "/".join(data_dir.split("/")[-1:])
    full_datapath = os.path.join("datasets", dataset, data_dir)
    return full_datapath


## Linear model metrics

In [3]:
def get_metrics_dataset_linear_model(model, model_path, dataset="synthetic_res_scbm"):
    full_data_path = get_data_dir_path_from_model_log(dataset, model, model_path)
    data_dir_name = full_data_path.split("/")[-1]
    linear_models_dir = os.path.join("experiments", "linear_head", dataset)
    for model_dir in os.listdir(linear_models_dir):
        model_name_start = data_dir_name + "_trueResUsed_False"
        if model_dir.startswith(model_name_start):
            linear_model_path = model_dir
            break
    with open(os.path.join(linear_models_dir, linear_model_path, "log.txt"), "r") as f:
        lines = f.readlines()
        for line in lines:
            if "Test" in line:
                test_line = line
                break
        y_accuracy = re.findall(r"Final Test Accuracy:\s*([0-9.]+)", test_line)[0]
        
    return float(y_accuracy)

# Independent covariance experiment



In [6]:
scbm_experiment_dir = os.path.join("experiments", "scbm", "synthetic_res_scbm")
scbm_res_experiment_dir = os.path.join("experiments", "scbm_residual", "synthetic_res_scbm")

models = [("scbm", "alpha_1.0_beta_1.0_rho_cr0_rho_cc0.0_rho_rr0.0_200_epochs_2026-06-02_16-26-50_303d0"),
 ("scbm_residual", "alpha_1.0_beta_1.0_rho_cr0_rho_cc0.0_rho_rr0.0_200_epochs_2026-06-02_16-22-14_ddf41")]


for model, model_path in models:
    metrics = test_metrics(model, model_path)
    print(f"Model: {model}")
    print(f"Test Metrics: {metrics}")



Model: scbm
Test Metrics: {'y_accuracy': 0.695, 'c_accuracy': 0.969, 'c_auc': 0.989}
Model: scbm_residual
Test Metrics: {'y_accuracy': 0.981, 'c_accuracy': 0.972, 'c_auc': 0.989}


### Dataset

In [ ]:
for model, model_path in models:
    data_path = get_data_dir_path_from_model_log(model, model_path)
    w_obs = torch.load(os.path.join(data_path, "train","w_obs.pt"))
    w_hid = torch.load(os.path.join(data_path, "train","w_hid.pt"))
    s_tr


In [7]:
root = "datasets/synthetic_res_scbm"
data1 = "cluster_a_1.0_b_1.0_rho_cr0_rho_cc0.0_rho_rr0.0_r_sparsity_0.5_c_sparsity_0.3_sigmax_0.5_seed_0"
data2 = "cluster_a_1.0_b_1.0_rho_cr0_rho_cc0.0_rho_rr0.0_r_sparsity_0.5_c_sparsity_0.3_sigmax_0.5_seed_0_v1"

datasets = [data1, data2]
splits = ["train", "val", "test"]

data_dict = {}

for data_dir in datasets:
    for split in splits:
        if not os.path.exists(os.path.join(root, data_dir, split)):
            print(f"Directory not found: {os.path.join(root, data_dir, split)}")
            continue
        print(f"Loading data from {data_dir} for split {split}")
        x_path = os.path.join(root, data_dir, split, "x.pt")
        print(f"Loading x from {x_path}")
        y_path = os.path.join(root, data_dir, split, "y.pt")
        s_path = os.path.join(root, data_dir, split, "s.pt")
        x = torch.load(x_path)
        y = torch.load(y_path)
        s = torch.load(s_path)
        print(x.shape, y.shape, s.shape)
        data_dict[(data_dir, split)] = (x, y, s)
        
for split in splits:
    x1, y1, s1 = data_dict[(data1, split)]
    x2, y2, s2 = data_dict[(data2, split)]
    # Check if x1 and x2 are the same
    if torch.allclose(x1, x2):
        print(f"x tensors are the same for split {split}")
    else:
        print(f"x tensors differ for split {split}")
    # Check if y1 and y2 are the same
    if torch.allclose(y1, y2):
        print(f"y tensors are the same for split {split}")
    else:
        print(f"y tensors differ for split {split}")
    # Check if s1 and s2 are the same
    if torch.allclose(s1, s2):
        print(f"s tensors are the same for split {split}")
    else:
        print(f"s tensors differ for split {split}")
    


Loading data from cluster_a_1.0_b_1.0_rho_cr0_rho_cc0.0_rho_rr0.0_r_sparsity_0.5_c_sparsity_0.3_sigmax_0.5_seed_0 for split train
Loading x from datasets/synthetic_res_scbm/cluster_a_1.0_b_1.0_rho_cr0_rho_cc0.0_rho_rr0.0_r_sparsity_0.5_c_sparsity_0.3_sigmax_0.5_seed_0/train/x.pt
torch.Size([30000, 100]) torch.Size([30000]) torch.Size([30000])
Loading data from cluster_a_1.0_b_1.0_rho_cr0_rho_cc0.0_rho_rr0.0_r_sparsity_0.5_c_sparsity_0.3_sigmax_0.5_seed_0 for split val
Loading x from datasets/synthetic_res_scbm/cluster_a_1.0_b_1.0_rho_cr0_rho_cc0.0_rho_rr0.0_r_sparsity_0.5_c_sparsity_0.3_sigmax_0.5_seed_0/val/x.pt
torch.Size([10000, 100]) torch.Size([10000]) torch.Size([10000])
Loading data from cluster_a_1.0_b_1.0_rho_cr0_rho_cc0.0_rho_rr0.0_r_sparsity_0.5_c_sparsity_0.3_sigmax_0.5_seed_0 for split test
Loading x from datasets/synthetic_res_scbm/cluster_a_1.0_b_1.0_rho_cr0_rho_cc0.0_rho_rr0.0_r_sparsity_0.5_c_sparsity_0.3_sigmax_0.5_seed_0/test/x.pt
torch.Size([10000, 100]) torch.Size(